# Hospital Readmission Analysis


## Project Overview

This notebook explores patterns in hospital readmissions within 30 days of discharge. We analyze patient demographics, diagnoses, length of stay, and discharge disposition to understand what factors drive avoidable readmissions.


## Learning Objectives

- Understand readmission rate variation across clinical and demographic segments
- Apply statistical tests to compare length of stay between readmitted and non-readmitted patients
- Identify high-risk patient profiles using exploratory analysis
- Communicate findings with visualizations grounded in computed results


## Business or Research Problem

Hospital readmissions are costly and often preventable. Identifying patient groups with elevated 30-day readmission risk can guide care coordination, discharge planning, and resource allocation.


## Why This Analysis Matters

Readmission rates are a key quality metric tied to hospital reimbursement under value-based care programs. Even a small reduction in preventable readmissions translates to significant cost savings and better patient outcomes.


## Dataset Overview

Synthetic dataset of 2500 inpatient visits. Variables include patient ID, admission and discharge dates, diagnosis category, length of stay, age group, gender, insurance type, discharge disposition, and 30-day readmission status.


## Dataset Source and License Notes

This dataset is entirely synthetic and generated with NumPy and pandas using seed=42. No real patient data is used. It is intended for educational and exploratory purposes only.


## Environment Setup


In [ ]:
import importlib
import subprocess
import sys

required = ['numpy', 'pandas', 'matplotlib', 'seaborn', 'scipy']
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        print(f'{pkg}: {mod.__version__}')
    except ImportError:
        print(f'{pkg} not found, installing...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])


## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)


## Configuration / Constants


In [ ]:
SEED = 42
N = 2500
READMISSION_RATE = 0.18
np.random.seed(SEED)


## Dataset Download or Loading

We generate 2500 synthetic patient visit records covering admissions from 2021 to 2023.


In [ ]:
np.random.seed(SEED)

diagnosis_categories = ['Cardiac', 'Respiratory', 'Diabetes', 'Orthopedic', 'Neurological', 'Gastrointestinal']
age_groups = ['18-34', '35-49', '50-64', '65-79', '80+']
genders = ['Male', 'Female']
insurance_types = ['Medicare', 'Medicaid', 'Private', 'Uninsured']
dispositions = ['Home', 'SNF', 'Rehab', 'Home Health', 'AMA']

admission_dates = pd.to_datetime('2021-01-01') + pd.to_timedelta(np.random.randint(0, 730, N), unit='D')
los_days = np.random.randint(1, 31, N)
discharge_dates = admission_dates + pd.to_timedelta(los_days, unit='D')

diagnosis = np.random.choice(diagnosis_categories, N)
age_group = np.random.choice(age_groups, N, p=[0.1, 0.15, 0.25, 0.3, 0.2])
gender = np.random.choice(genders, N)
insurance = np.random.choice(insurance_types, N, p=[0.35, 0.25, 0.30, 0.10])
disposition = np.random.choice(dispositions, N, p=[0.45, 0.20, 0.15, 0.15, 0.05])

# Readmission influenced by age, diagnosis and disposition
base_prob = np.full(N, READMISSION_RATE)
base_prob[diagnosis == 'Cardiac'] += 0.05
base_prob[diagnosis == 'Respiratory'] += 0.04
base_prob[age_group == '80+'] += 0.06
base_prob[age_group == '65-79'] += 0.03
base_prob[disposition == 'AMA'] += 0.10
base_prob[insurance == 'Uninsured'] += 0.05
base_prob = np.clip(base_prob, 0, 1)
readmitted = (np.random.rand(N) < base_prob).astype(int)

df = pd.DataFrame({
    'patient_id': [f'P{i:05d}' for i in range(N)],
    'admission_date': admission_dates,
    'discharge_date': discharge_dates,
    'diagnosis_category': diagnosis,
    'los_days': los_days,
    'age_group': age_group,
    'gender': gender,
    'insurance_type': insurance,
    'discharge_disposition': disposition,
    'readmitted_30days': readmitted
})

print(df.shape)
df.head()


## Data Validation Checks

We verify expected shapes, data types, missing values, and value ranges before analysis.


In [ ]:
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nLOS range:', df['los_days'].min(), '-', df['los_days'].max())
print('Readmission values:', df['readmitted_30days'].unique())
print('Overall readmission rate:', round(df['readmitted_30days'].mean(), 4))


## Data Cleaning

No missing values are expected in synthetic data, but we confirm and document any anomalies.


In [ ]:
# Confirm no negative LOS or invalid dates
assert (df['los_days'] >= 1).all(), 'LOS must be >= 1'
assert (df['discharge_date'] >= df['admission_date']).all(), 'Discharge before admission'
assert df['readmitted_30days'].isin([0, 1]).all(), 'Invalid readmission flag'
print('All validation checks passed.')
df['admission_month'] = df['admission_date'].dt.month
df['admission_year'] = df['admission_date'].dt.year
df['admission_quarter'] = df['admission_date'].dt.quarter


## Exploratory Data Analysis

We begin with high-level summary statistics to understand the dataset before drilling into specific patterns.


In [ ]:
print(df.describe())
print('\nReadmission rate by year:')
print(df.groupby('admission_year')['readmitted_30days'].mean().round(4))


## Univariate Analysis

Length of stay distribution tells us about acuity and resource use. We visualize its overall spread.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df['los_days'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Length of Stay')
axes[0].set_xlabel('LOS (days)')
axes[0].set_ylabel('Count')

readmission_counts = df['readmitted_30days'].value_counts()
axes[1].bar(['Not Readmitted', 'Readmitted'], readmission_counts.values, color=['steelblue', 'tomato'])
axes[1].set_title('Readmission Count')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()


LOS is roughly uniform between 1 and 30 days reflecting synthetic generation. The readmission count shows roughly 18% of patients are readmitted.


## Bivariate / Multivariate Analysis

Now we look at readmission rate by key categorical variables to identify differential risk.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

diag_rate = df.groupby('diagnosis_category')['readmitted_30days'].mean().sort_values(ascending=False)
diag_rate.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Readmission Rate by Diagnosis')
axes[0].set_ylabel('Readmission Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

age_rate = df.groupby('age_group')['readmitted_30days'].mean().reindex(age_groups)
age_rate.plot(kind='bar', ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Readmission Rate by Age Group')
axes[1].set_ylabel('Readmission Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()


Cardiac and Respiratory patients show the highest readmission rates. Older age groups (65-79, 80+) exhibit elevated risk consistent with comorbidity burden.


## Feature-Specific Insights

Insurance type and discharge disposition may reflect access to follow-up care, which in turn affects readmission likelihood.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ins_rate = df.groupby('insurance_type')['readmitted_30days'].mean().sort_values(ascending=False)
ins_rate.plot(kind='bar', ax=axes[0], color='mediumseagreen', edgecolor='white')
axes[0].set_title('Readmission Rate by Insurance Type')
axes[0].set_ylabel('Readmission Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

disp_rate = df.groupby('discharge_disposition')['readmitted_30days'].mean().sort_values(ascending=False)
disp_rate.plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Readmission Rate by Discharge Disposition')
axes[1].set_ylabel('Readmission Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()


Uninsured patients and AMA (Against Medical Advice) discharges show the highest readmission rates, reflecting care gaps and incomplete treatment.


## Statistical Checks

We use an independent samples t-test to compare mean LOS between readmitted and non-readmitted patients.


In [ ]:
los_readmitted = df[df['readmitted_30days'] == 1]['los_days']
los_not_readmitted = df[df['readmitted_30days'] == 0]['los_days']

t_stat, p_val = stats.ttest_ind(los_readmitted, los_not_readmitted)
mean_readmitted = los_readmitted.mean()
mean_not_readmitted = los_not_readmitted.mean()

print(f'Mean LOS (Readmitted): {mean_readmitted:.2f} days')
print(f'Mean LOS (Not Readmitted): {mean_not_readmitted:.2f} days')
print(f'T-statistic: {t_stat:.3f}, P-value: {p_val:.4f}')
if p_val < 0.05:
    print('Result: Statistically significant difference in LOS (p < 0.05)')
else:
    print('Result: No statistically significant difference in LOS (p >= 0.05)')


## Visual Analysis

Seasonal admission patterns can reveal periods of elevated healthcare demand.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

monthly = df.groupby(['admission_year', 'admission_month'])['readmitted_30days'].mean().reset_index()
for year in df['admission_year'].unique():
    subset = monthly[monthly['admission_year'] == year]
    axes[0].plot(subset['admission_month'], subset['readmitted_30days'], marker='o', label=str(year))
axes[0].set_title('Monthly Readmission Rate by Year')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Readmission Rate')
axes[0].legend(title='Year')

los_readmit = df.groupby('readmitted_30days')['los_days'].apply(list)
axes[1].boxplot([los_readmit[0], los_readmit[1]], labels=['Not Readmitted', 'Readmitted'])
axes[1].set_title('LOS by Readmission Status')
axes[1].set_ylabel('LOS (days)')

plt.tight_layout()
plt.show()


Readmission rates show modest seasonal variation. The LOS box plots illustrate the spread of stay duration for each group.


## Key Findings

We summarize the most important findings derived directly from computed statistics.


In [ ]:
overall_rate = df['readmitted_30days'].mean()
top_diag = diag_rate.idxmax()
top_diag_rate = diag_rate.max()
top_age = age_rate.idxmax()
top_age_rate = age_rate.max()
top_ins = ins_rate.idxmax()
top_ins_rate = ins_rate.max()
top_disp = disp_rate.idxmax()
top_disp_rate = disp_rate.max()

print(f'Overall 30-day readmission rate: {overall_rate:.1%}')
print(f'Highest readmission rate by diagnosis: {top_diag} ({top_diag_rate:.1%})')
print(f'Highest readmission rate by age group: {top_age} ({top_age_rate:.1%})')
print(f'Highest readmission rate by insurance: {top_ins} ({top_ins_rate:.1%})')
print(f'Highest readmission rate by disposition: {top_disp} ({top_disp_rate:.1%})')
print(f'Mean LOS readmitted vs not: {mean_readmitted:.2f} vs {mean_not_readmitted:.2f} days')
print(f'T-test p-value: {p_val:.4f} — {"significant" if p_val < 0.05 else "not significant"}')


## Limitations

- Synthetic data does not capture real clinical complexity or comorbidity interactions.
- Readmission probability was assigned using additive rules rather than a calibrated model.
- No causal inference is possible from cross-sectional exploratory analysis.
- Seasonal patterns reflect random sampling rather than true epidemiological cycles.


## Recommendations / Next Steps

- Build a predictive readmission model using logistic regression or gradient boosting.
- Integrate social determinants of health features (housing, transport) for richer risk scoring.
- Apply survival analysis to model time-to-readmission rather than binary 30-day flag.
- Validate findings on real hospital claims data under appropriate data governance.


## Common Mistakes

- Confusing readmission rate with readmission count — always normalize by group size.
- Ignoring discharge disposition in readmission models despite its strong predictive value.
- Using accuracy as the sole metric for a heavily imbalanced readmission dataset.
- Failing to account for seasonal variation in admission volumes when computing rates.


## Mini Challenge / Exercises

1. Compute readmission rate separately for each combination of age group and insurance type.
2. Create a heatmap of readmission rate by diagnosis category and discharge disposition.
3. Test whether cardiac patients have a significantly different LOS than respiratory patients.
4. Add a 'high_risk' flag for patients in the top 20% of readmission probability and analyze their profile.


## Final Summary / Key Takeaways

- The overall 30-day readmission rate is approximately 18% with meaningful variation by clinical and demographic group.
- Cardiac and respiratory diagnoses, elderly age groups, uninsured status, and AMA discharges are the strongest risk indicators.
- LOS alone may not statistically separate readmitted from non-readmitted patients in this synthetic dataset.
- Targeted interventions for high-risk segments — particularly discharge support for cardiac and elderly patients — are supported by the analysis.
